# Lesson 06
Run lessons 1–5 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第6课 组合配置与市场状态风险

本课把第2—5课信号转为可执行权重，比较等权、逆波动、收缩协方差最小方差、状态过滤及风险匹配对照。固定风险资产预算0.8，余款现金；设单股、行业、总敞口、净敞口与换手限制，并输出不可行回退原因。

风险预测只用决策收盘以前的数据：历史、EWMA、滚动GARCH(1,1)，下一开盘生效。GARCH为收盘收益风险代理，不能冒充开盘交易收益的精确条件分布。VaR/ES用账本同口径历史净收益生成下一期阈值，报告突破、聚集与有限样本尾部均值。多空单独展示净/总/借券约束，不混用只做多优化条件。

所有组合共用第二课自融资账本：扣费后目标仓位、逐腿交易费、现金、借券与融资各记一次。状态过滤另与训练期冻结的等波动缩放基线比较；不会只凭更低回撤就断言更有效。

运行环境保留TPU：Pandas/SciPy/arch和SB3使用主机CPU；第8课JAX PPO梯度、第9课蒙特卡洛实际放在TPU。设备验证失败就明确停止加速实验，不把设置名称当成运行证明。所有数据窗2014—2022都是开发研究，不宣称未观察的最终留出。

参考：[arch预测](https://arch.readthedocs.io/en/stable/univariate/forecasting.html)、[JAX设备](https://docs.jax.dev/en/latest/installation.html)。


In [ ]:
# 06.01 扩展依赖、TPU实际计算与重现日志
import sys,subprocess,importlib.metadata as im,io,json,time,hashlib,warnings
subprocess.check_call([sys.executable,'-m','pip','install','-q','arch==7.2.0','gymnasium==1.2.2','stable-baselines3==2.7.0','QuantLib==1.40','nbformat==5.10.4'])
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display,Markdown
import jax,jax.numpy as jnp
DEVICES=jax.devices(); TPU_DEVICES=[d for d in DEVICES if d.platform=='tpu']
print('JAX',jax.__version__,'actual devices',DEVICES)
assert TPU_DEVICES,'User requested TPU: select TPU runtime and reconnect; do not label CPU results as TPU'
TPU=TPU_DEVICES[0]
with jax.default_device(TPU):
    a=jnp.arange(1024,dtype=jnp.float32).reshape(32,32); device_checksum=float(jax.jit(lambda x:jnp.sum(x@x.T))(a).block_until_ready())
print('TPU real matrix computation checksum',device_checksum)
L6_RUNTIME={'started_utc':pd.Timestamp.now(tz='UTC').isoformat(),'python':sys.version,'jax':jax.__version__,
 'devices':[str(d) for d in DEVICES],'TPU_checksum':device_checksum,
 'versions':{p:im.version(p) for p in ['numpy','pandas','scipy','arch','gymnasium','stable-baselines3','QuantLib']}}
print(json.dumps(L6_RUNTIME,ensure_ascii=False,indent=2))
print('Prerequisite variables present:',{k:k in globals() for k in ['OPEN','CLOSE','D5','L5_MODELS','l2_simulate_multi']})


In [ ]:
# 06.02 因果风险特征、收缩协方差与约束权重生成器
assert all(k in globals() for k in ['OPEN','CLOSE','D5','L5_MODELS','l2_simulate_multi']),'请先运行前五课'
from scipy.optimize import minimize
from scipy.stats import norm,chi2
from sklearn.covariance import LedoitWolf
from arch import arch_model
R6=CLOSE[ASSETS].pct_change(fill_method=None)
EXEC6=OPEN.loc['2014-02-01':'2022-12-29'].index
PREV6=pd.Series(OPEN.index,index=OPEN.index).shift(1).reindex(EXEC6)
GROUP6={g:[ASSETS.index(a) for a in ASSETS if UNIVERSE[a][1]==g] for g in sorted({UNIVERSE[a][1] for a in ASSETS})}
CFG6={'risk_budget':.8,'single_cap':.15,'industry_cap':.35,'gross_cap':.8,'net_cap':.8,'turnover_cap':.25,
 'cov_window':252,'cov_refit':21,'commission_bps':3,'slippage_bps':2,'state_train_end':'2019-01-01','EWMA_lambda':.94}
FALLBACK6=[]
def caps_ok(w,budget=.8,tol=1e-7):
    return abs(np.sum(w)-budget)<tol and np.min(w)>=-tol and np.max(w)<=.15+tol and all(np.sum(w[j])<=.35+tol for j in GROUP6.values())
def feasible_budget(v,budget=.8):
    if budget==0:return np.zeros(len(ASSETS))
    v=np.asarray(v,float); v=np.nan_to_num(v,nan=0,posinf=0,neginf=0)
    objective=lambda w:np.sum((w-v)**2)
    con=[{'type':'eq','fun':lambda w:w.sum()-budget}]+[{'type':'ineq','fun':lambda w,j=j:.35-w[j].sum()} for j in GROUP6.values()]
    res=minimize(objective,np.ones(len(v))*budget/len(v),bounds=[(0,.15)]*len(v),constraints=con,method='SLSQP',options={'ftol':1e-12,'maxiter':150})
    if not res.success or not caps_ok(res.x,budget):
        FALLBACK6.append({'where':'projection','reason':res.message,'budget':budget}); fallback=np.ones(len(v))*budget/len(v)
        assert caps_ok(fallback,budget); return fallback
    return res.x
SIG6=pd.DataFrame(index=CLOSE.index)
SIG6['hist_vol']=CLOSE.SPY.pct_change(fill_method=None).rolling(63).std()*np.sqrt(252)
SIG6['ewma_vol']=np.sqrt(CLOSE.SPY.pct_change(fill_method=None).pow(2).ewm(alpha=1-.94,adjust=False,min_periods=63).mean()*252)
SIG6['trend']=CLOSE.SPY/CLOSE.SPY.rolling(200).mean()-1
state_threshold=SIG6.loc['2014-01-01':'2018-12-31','ewma_vol'].quantile(.8)
SIG6['state_active']=(SIG6.ewma_vol<=state_threshold)&(SIG6.trend>0)
SIG6['GARCH_vol']=np.nan; garch_log=[]
for i in range(756,len(CLOSE),21):
    past=100*CLOSE.SPY.pct_change(fill_method=None).iloc[i-756:i+1].dropna()
    try:
        fitted=arch_model(past,mean='Zero',vol='GARCH',p=1,q=1,rescale=False).fit(disp='off',show_warning=False)
        omega,alpha,beta=[float(fitted.params[k]) for k in ['omega','alpha[1]','beta[1]']]
        variance=float(fitted.conditional_volatility.iloc[-1]**2)
        for j in range(i,min(i+21,len(CLOSE))):
            forecast=omega+alpha*(100*(CLOSE.SPY.iloc[j]/CLOSE.SPY.iloc[j-1]-1))**2+beta*variance
            SIG6.iloc[j,SIG6.columns.get_loc('GARCH_vol')]=np.sqrt(forecast*252)/100
            variance=forecast
        garch_log.append({'fit_end':CLOSE.index[i],'first_forecast_for':CLOSE.index[i+1] if i+1<len(CLOSE) else pd.NaT,'success':fitted.convergence_flag==0,'alpha_plus_beta':alpha+beta})
    except Exception as e:garch_log.append({'fit_end':CLOSE.index[i],'success':False,'error':str(e)})
# 日内决策生成，下日开盘执行。风险模型原始参数每21日重估，期间仅用已经实现的残差递推。
W6_DEC={n:pd.DataFrame(0.,index=CLOSE.index,columns=ASSETS) for n in ['equal','inverse_vol','shrink_minvar','signal_tilt']}
COV6={}; COV_ASOF6={}; last_cov=None; last_fit=None
panel6=PANEL.set_index(['date','ticker'])
for i,date in enumerate(CLOSE.index):
    if date<pd.Timestamp('2014-01-01'):continue
    hist=R6.loc[:date].tail(252).dropna()
    if len(hist)<200:continue
    if last_cov is None or i%21==0:
        last_cov=LedoitWolf().fit(hist.to_numpy()).covariance_;last_fit=date
    COV6[date]=last_cov.copy();COV_ASOF6[date]=last_fit
    eq=np.ones(len(ASSETS))*.8/len(ASSETS)
    iv=1/np.sqrt(np.maximum(np.diag(last_cov),1e-12));iv=feasible_budget(.8*iv/iv.sum())
    con=[{'type':'eq','fun':lambda w:w.sum()-.8}]+[{'type':'ineq','fun':lambda w,j=j:.35-w[j].sum()} for j in GROUP6.values()]
    if i%21==0 or 'mv' not in locals():
        result=minimize(lambda w:float(w@last_cov@w)*1e5,eq,bounds=[(0,.15)]*len(ASSETS),constraints=con,method='SLSQP',options={'ftol':1e-10,'maxiter':150})
        if result.success and caps_ok(result.x):mv=result.x
        else:mv=iv.copy();FALLBACK6.append({'where':'minvar','date':str(date),'reason':result.message})
    try:
        scores=panel6.loc[date].reindex(ASSETS)[[f+'_z' for f in FACTOR_NAMES]].mean(axis=1).fillna(0).to_numpy()
    except KeyError:scores=np.zeros(len(ASSETS))
    tilt=feasible_budget(eq*(1+.3*np.tanh(scores)))
    for name,w in [('equal',eq),('inverse_vol',iv),('shrink_minvar',mv),('signal_tilt',tilt)]:W6_DEC[name].loc[date]=w
print('Frozen training-only state threshold',state_threshold);display(pd.DataFrame(garch_log).tail());print('fallback events',len(FALLBACK6))
print('06.02 PASS: covariance fit dates and state threshold recorded')


In [ ]:
# 06.03 同一账本的风险配置、换手限制与风险匹配对照
W6_DEC['state_filtered']=W6_DEC['shrink_minvar'].mul(SIG6.state_active.astype(float)*.75+.25,axis=0)
DESIRED6={n:w.shift(1).reindex(EXEC6).fillna(0) for n,w in W6_DEC.items()}
def limited_targets(opens,desired,turnover_cap=.25,cost=.0005,initial=10000.):
    cash=initial;units=np.zeros(desired.shape[1]);rows=[];audits=[]
    def solve(w,old,E):
        post=E
        for _ in range(40):post=E-np.abs(w*post-old).sum()*cost
        return post,w*post-old
    for count,(d,target) in enumerate(desired.iterrows()):
        p=opens.loc[d].to_numpy();nxt=opens.shift(-1).loc[d].to_numpy();old=units*p;E=cash+old.sum();oldw=old/E
        # 若行情漂移使权重超过上限，先做最小比例缩减；超出换手限制的强制降险单独登记。
        repair=min(1.,.8/max(oldw.sum(),1e-12),.15/max(oldw.max(),1e-12),*[.35/max(oldw[j].sum(),1e-12) for j in GROUP6.values()])
        safe=oldw*repair;want=target.to_numpy(float);post,delta=solve(want,old,E);limited=False
        if count>0 and np.abs(delta).sum()/E>turnover_cap:
            low,high=0.,1.
            for _ in range(35):
                mid=(low+high)/2;candidate=safe+mid*(want-safe);p0,d0=solve(candidate,old,E)
                if np.abs(d0).sum()/E<=turnover_cap:low=mid
                else:high=mid
            want=safe+low*(want-safe);post,delta=solve(want,old,E);limited=True
        turn=np.abs(delta).sum()/E
        forced=bool(count>0 and turn>turnover_cap+1e-6)
        fees=np.abs(delta).sum()*cost;units+=delta/p;cash-=delta.sum()+fees
        rows.append(want.copy());audits.append({'date':d,'desired_budget':target.sum(),'executed_budget':want.sum(),'turnover':turn,'limited':limited,'forced_risk_override':forced,'startup':count==0})
        assert cash+units@nxt>0
    return pd.DataFrame(rows,index=desired.index,columns=desired.columns),pd.DataFrame(audits).set_index('date')
L6_TARGETS={};L6_LEDGERS={};L6_TRADES={};L6_LIMIT_LOG={}
def run6(name,desired):
    target,audit=limited_targets(OPEN[ASSETS],desired)
    x,t=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
    assert np.allclose(x.turnover,audit.turnover,atol=1e-7)
    L6_TARGETS[name]=target;L6_LEDGERS[name]=x;L6_TRADES[name]=t;L6_LIMIT_LOG[name]=audit
for name,w in DESIRED6.items():run6(name,w)
train_period=slice('2014-02-01','2018-12-31')
matched_scale=min(1.,L6_LEDGERS['state_filtered'].loc[train_period].net_return.std()/L6_LEDGERS['shrink_minvar'].loc[train_period].net_return.std())
run6('risk_matched_unfiltered',DESIRED6['shrink_minvar']*matched_scale)
RESULT6=pd.DataFrame([{'model':n,'period':period,**ledger_stats(x.loc[a:b]),'realized_vol':x.loc[a:b].net_return.std()*np.sqrt(252),'avg_gross':x.loc[a:b].gross_exposure.mean()}
 for n,x in L6_LEDGERS.items() for period,a,b in [('fit','2014-02-01','2018-12-31'),('validation','2019-01-01','2020-12-31'),('development_eval','2021-01-01','2022-12-29')]]).set_index(['period','model'])
print('Risk matching scale frozen from 2014–2018',matched_scale);display(RESULT6)
print('实际评估波动不保证完全相等；风险匹配仅训练期冻结，不按评估结果反向缩放。')
display(pd.DataFrame({n:a[['limited','forced_risk_override']].sum() for n,a in L6_LIMIT_LOG.items()}).T)
fig,ax=plt.subplots(2,2,figsize=(14,8));pd.DataFrame({n:x.wealth for n,x in L6_LEDGERS.items()}).plot(ax=ax[0,0],title='Constrained account net wealth')
pd.DataFrame({n:x.drawdown for n,x in L6_LEDGERS.items()}).plot(ax=ax[0,1],title='Account drawdowns')
SIG6.loc['2014':'2022',['hist_vol','ewma_vol','GARCH_vol']].plot(ax=ax[1,0],title='Past-only close-return volatility forecasts')
L6_TARGETS['state_filtered'].plot.area(ax=ax[1,1],legend=False,title='Realized constrained target weights');plt.tight_layout();plt.show()
print('06.03 PASS: self-financing replay and turnover overlay agree')


In [ ]:
# 06.04 风险贡献、VaR/ES突破聚集、危机压力与多空约束
# 使用GARCH预测作为风险缩放候选：预算只能下降，不借预测提高总杠杆。
garch_multiplier=(.15/SIG6.GARCH_vol.clip(lower=.01)).clip(lower=.25,upper=1).fillna(.25)
garch_desired=W6_DEC['shrink_minvar'].mul(garch_multiplier,axis=0).shift(1).reindex(EXEC6).fillna(0)
run6('GARCH_risk_scaled',garch_desired)
TAIL6=[];TAIL_SERIES6={};CONTRIB6=[]
def independence_lr(hit):
    a=np.asarray(hit,dtype=int);counts=np.array([[np.sum((a[:-1]==i)&(a[1:]==j)) for j in [0,1]] for i in [0,1]],float)
    def ll(n,p):return sum(n[j]*np.log(np.clip([1-p,p][j],1e-12,1)) for j in [0,1])
    p=counts[:,1].sum()/max(counts.sum(),1);p0=counts[0,1]/max(counts[0].sum(),1);p1=counts[1,1]/max(counts[1].sum(),1)
    lr=2*(ll(counts[0],p0)+ll(counts[1],p1)-ll(counts.sum(axis=0),p))
    return lr,float(chi2.sf(max(lr,0),1)),int(np.sum((a[1:]==1)&(a[:-1]==1)))
for name,x in L6_LEDGERS.items():
    loss=-x.net_return;var=loss.shift(1).rolling(250,min_periods=250).quantile(.95)
    es=loss.shift(1).rolling(250,min_periods=250).apply(lambda z:np.mean(z[z>=np.quantile(z,.95)]),raw=True)
    z=pd.DataFrame({'loss':loss,'VaR95':var,'ES95':es}).dropna();z=z.loc['2019-01-01':];z['breach']=z.loss>z.VaR95
    lr,p,consecutive=independence_lr(z.breach)
    TAIL6.append({'model':name,'n':len(z),'breaches':int(z.breach.sum()),'breach_rate':z.breach.mean(),'expected_rate':.05,
      'mean_VaR':z.VaR95.mean(),'mean_ES':z.ES95.mean(),'mean_exceedance_loss':z.loc[z.breach,'loss'].mean(),
      'LR_independence':lr,'asymptotic_p':p,'consecutive_breaches':consecutive,'tail_n_per_window':'about 13; uncertain'})
    TAIL_SERIES6[name]=z
    for d in x.index[::21]:
        decision=PREV6.loc[d];cov=COV6[decision];w=L6_TARGETS[name].loc[d].to_numpy();variance=w@cov@w
        rc=w*(cov@w)/max(variance,1e-15)
        for a,v in zip(ASSETS,rc):CONTRIB6.append({'date':d,'model':name,'instrument':a,'variance_fraction':v})
TAIL_REPORT6=pd.DataFrame(TAIL6).set_index('model');display(TAIL_REPORT6)
RC6=pd.DataFrame(CONTRIB6);display(RC6.groupby(['model','instrument']).variance_fraction.mean().unstack())
STRESS6=pd.DataFrame([{'model':n,'window':w,**ledger_stats(x.loc[a:b])} for n,x in L6_LEDGERS.items()
 for w,a,b in [('2018Q4','2018-10-01','2018-12-31'),('COVID','2020-02-19','2020-04-30'),('2022','2022-01-01','2022-12-29')]]).set_index(['window','model']);display(STRESS6)
# 同波动、相关性提升到0.9的情景：这是假设压力，不能冒充历史观测。
date=PREV6.iloc[-1];cov=COV6[date];sd=np.sqrt(np.diag(cov));corr_stress=.9*np.ones_like(cov)+.1*np.eye(len(ASSETS));shock=np.outer(sd,sd)*corr_stress
w=L6_TARGETS['shrink_minvar'].iloc[-1].to_numpy()
print('Correlation-stress annual vol',np.sqrt(w@shock@w*252),'vs estimated',np.sqrt(w@cov@w*252))
# 多空独立演示：gross .8/net0/单股.1；借券不可得时关闭账户，假设必须标记。
LS=pd.DataFrame(0.,index=EXEC6,columns=ASSETS)
for d in EXEC6:
    dec=PREV6.loc[d]
    if dec not in panel6.index.get_level_values('date'):continue
    g=panel6.loc[dec].reindex(ASSETS);score=g[[f+'_z' for f in FACTOR_NAMES]].mean(axis=1).fillna(0).sort_values(kind='stable')
    LS.loc[d,score.index[:4]]=-.1;LS.loc[d,score.index[-4:]]=.1
LS_LEDGER,LS_TRADES=l2_simulate_multi(OPEN[ASSETS],LS,commission=3,slippage=2,borrow_rate=.03,funding=.05,initial=10000)
available=pd.DataFrame(True,index=LS.index,columns=ASSETS);test_date=LS.index[len(LS)//2];short_name=LS.loc[test_date].idxmin();available.loc[test_date,short_name]=False
LS_BLOCKED,_=l2_simulate_multi(OPEN[ASSETS],LS,commission=3,slippage=2,borrow_rate=.03,funding=.05,available=available,initial=10000)
assert LS_BLOCKED.loc[test_date,'blocked_by_borrow'] and abs(LS_BLOCKED.loc[test_date,'gross_exposure'])<1e-8
print('Long/short scenario, not observed historical borrow');display(pd.DataFrame({'long_short':ledger_stats(LS_LEDGER)}).T)
# 同一个成本账户的完整月收盘权益归因；丢弃不完整最后月。
x=L6_LEDGERS['shrink_minvar'];eqclose=x.cash+sum(x['units_'+a]*CLOSE[a].reindex(x.index) for a in ASSETS)
monthly=eqclose.groupby(eqclose.index.to_period('M')).last();ret=monthly.pct_change(fill_method=None)
ff6=pd.concat([ret.rename('strategy'),FF],axis=1).loc['2014-03':'2022-11'].dropna()
fit6=sm.OLS(ff6.strategy-ff6.RF,sm.add_constant(ff6[['Mkt-RF','SMB','HML','RMW','CMA','Mom']])).fit(cov_type='HAC',cov_kwds={'maxlags':3})
display(pd.DataFrame({'loading':fit6.params,'HAC_t':fit6.tvalues}))
fig,ax=plt.subplots(1,2,figsize=(13,4));TAIL_SERIES6['shrink_minvar'][['loss','VaR95','ES95']].plot(ax=ax[0],title='Lagged same-account historical VaR and ES')
RC6[RC6.model.eq('shrink_minvar')].pivot(index='date',columns='instrument',values='variance_fraction').plot.area(ax=ax[1],legend=False,title='Shrink covariance variance contributions');plt.tight_layout();plt.show()
L6_CHECKS=[]
def check6(name,ok):
    assert bool(ok),name;L6_CHECKS.append({'check':name,'status':'PASS'})
check6('TPU executed',len(TPU_DEVICES)>0 and np.isfinite(device_checksum))
check6('covariance asof before open',all(COV_ASOF6[d]<e for e,d in PREV6.items()))
check6('frozen state threshold uses only fit dates',SIG6.loc['2014':'2018'].index.max()<TRAIN_END)
check6('long-short net and gross',(LS.sum(axis=1).abs()<1e-8).all() and (LS.abs().sum(axis=1)<=.800001).all())
check6('unavailable borrow closes portfolio',LS_BLOCKED.loc[test_date,'blocked_by_borrow'] and abs(LS_BLOCKED.loc[test_date,'gross_exposure'])<1e-8)
check6('ES is tail loss at or above VaR',all((z.ES95>=z.VaR95-1e-10).all() for z in TAIL_SERIES6.values()))
for name,x in L6_LEDGERS.items():
    w=L6_TARGETS[name];audit=L6_LIMIT_LOG[name]
    check6(name+' same window',x.index.equals(EXEC6))
    check6(name+' self finance',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost))
    check6(name+' positive costs',(x.transaction_cost>=0).all())
    check6(name+' position constraints',(w>=-1e-8).all().all() and w.max().max()<=.150001 and w.sum(axis=1).max()<=.800001 and all(w.iloc[:,j].sum(axis=1).max()<=.350001 for j in GROUP6.values()))
    check6(name+' turnover cap or recorded forced risk',((audit.turnover<=.250001)|audit.startup|audit.forced_risk_override).all())
    check6(name+' compound wealth',np.allclose(x.wealth,(1+x.net_return).cumprod()))
L6_VALIDATION=pd.DataFrame(L6_CHECKS);display(L6_VALIDATION)
L6_CARD={'lesson':6,'status':'verified','checks':len(L6_CHECKS),'constraints':CFG6,'risk_match_scale':matched_scale,
 'limitations':['fixed survivors','daily weight maintenance','risk vol proxy uses closes','ES small tail sample','historical borrow assumed','forced risk overrides explicitly recorded','all historical data already development']}
print('L6 FINAL PASS',json.dumps(L6_CARD,ensure_ascii=False));
